# Notebook 02 — TA CRM QC and pH-standard QC

**Role in the pipeline:** this is the *first* notebook on the critical path.
Its `derived.csv` (per sheet) is the file that Stage 1A reads.

```text
oa_prelim_data.xlsx
   │
   └──► 02_ta_ph_qc.ipynb  (THIS NOTEBOOK)
           ├─ TA CRM correction
           ├─ pH-standard correction
           └─ derived.csv  ──►  04_stage1a / 05_stage1b / ... / 08_stage4
```

**What it does, per sheet**

1. Detects TA Certified Reference Material (CRM) rows, computes
   `certified − measured` Total Alkalinity, applies the SOP rules
   (`NO_ADJUST` / `ADJUST` / `FAIL` / `INSUFFICIENT_DATA`), and writes
   corrected TA back to the dataframe.
2. Detects pH-standard rows (TRIS / AMP / BIS), interpolates expected pH at
   the measured temperature, computes the residual, classifies as
   `OK` / `WARN` / `FAIL`, and (optionally) corrects sample pH.
3. Writes a `derived.csv` (the QC-corrected sheet), per-QC tables, JPEG
   plots, and one-page markdown reports.
4. Records every input, parameter, output and version in `logs/manifest.json`.

All the heavy QC math lives in `oa_qc_ta_ph.py`; this notebook is a thin
orchestrator. See `02_ta_ph_qc.README.md` for the design rationale and
the source citations behind each choice.


## Parameters

Single tagged `parameters` cell — override from the command line via
Papermill (`papermill 02_ta_ph_qc.ipynb run.ipynb -p XLSX_PATH "..."`) or
edit and Restart-and-Run-All in Jupyter.

The block is long because the QC has many knobs. Sections are grouped by
purpose. If you only want the defaults, you usually only need to change
`XLSX_PATH` and possibly `SHEET`.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================
# All explanations live in the markdown cell directly below this one.
# Keeping this cell free of inline trailing comments avoids the
# Papermill "unknown parameter" warnings that those comments trigger.

# --- I/O -------------------------------------------------------------
XLSX_PATH = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_prelim_data.xlsx"
OUT_DIR = None
SHEET = 0
CONFIG_PATH = "None"

# --- Display / output toggles ---------------------------------------
PREVIEW_ROWS = 15
HTML_MAX_ROWS = None
OPEN_HTML = False
WRITE_DERIVED_CSV = True
WRITE_HTML_TABLE = True
WRITE_PLOTS = True

# --- Column names expected in each sheet ----------------------------
TA_COL = "ta"
PH_COL = "pH_lab"
PH_TEMP_COL = "temp_lab"
SAMPLE_TAG_COL = "sample_tag"
CRM_OR_SAMPLE_COL = "crm_or_sample"

# --- TA CRM QC ------------------------------------------------------
CRM_CORRECT_TA = True
CRM_BATCH = "213"
# AUDIT FIX N-1: path to the authoritative certified CRM values file
# (configs/crm_certified_values.yaml, transcribed from the NOAA OCADS
# Dickson CRM batch table). When None, the corrected in-code fallback
# table in oa_pipeline.qc_ta_ph is used. Prefer the YAML file.
CRM_VALUES_CONFIG = "configs/crm_certified_values.yaml"
CRM_TA_OVERRIDE = None
CRM_TAG_PREFIX = "RM"
ALLOW_CRM_FLAG_COL = False
NO_REQUIRE_TA_FOR_CRM = False
GROUP_BY = None
MIN_CRM_N = 2
TA_MAD_K = 3.5
TA_MAX_ABS_DIFF = 20.0
CORRECT_CRM_TOO = False
TA_NO_ADJUST = 2.0
TA_REJECT = 20.0
TA_PLOT_TITLE = "RM Alkalinity Difference"
TA_PLOT_NO_LABELS = False

# --- pH standard QC -------------------------------------------------
PHSTD_QC = True
PH_BUFFER = "tris"
PHSTD_TAG_PREFIX = "tris"
PHSTD_CORRECT_SAMPLES = False
MIN_PHSTD_N = 2
PH_MAD_K = 3.5
PH_MAX_ABS_DIFF = 0.03
PH_OK = 0.02
PH_WARN = 0.05
PH_PLOT_TITLE = "TRIS pH Difference"
PH_PLOT_NO_LABELS = False


### Parameter reference

The cell above is the single Papermill-tagged `parameters` cell. Keeping it
free of inline trailing comments avoids spurious *unknown parameter*
warnings from Papermill. The per-parameter notes that used to live next to
each assignment are reproduced below.

**I/O**

- `XLSX_PATH` — full path to the source `.xlsx` workbook.
- `OUT_DIR` — output directory. `None` means use `"<workbook_stem>__qc_outputs/"` next to the workbook.
- `SHEET` — `"0"`, a sheet name, or `"all"`.
- `CONFIG_PATH` — path to a JSON/YAML config to override defaults. The literal string `"None"` (or `None`) means "use built-in defaults".

**Display / output toggles**

- `HTML_MAX_ROWS` — `None` means write all rows to the HTML table.
- `OPEN_HTML` — open the HTML table in a browser after writing.
- `WRITE_DERIVED_CSV` / `WRITE_HTML_TABLE` / `WRITE_PLOTS` — output toggles.

**Column names**

- `CRM_OR_SAMPLE_COL` — set to `None` if the workbook has no flag column.

**TA CRM QC**

- `CRM_BATCH` — must be a key in `CRM_CERTIFIED_TA` inside `oa_qc_ta_ph.py`.
- `CRM_TA_OVERRIDE` — numeric value overrides the certified TA; `None` uses the certified value for `CRM_BATCH`.
- `CRM_TAG_PREFIX` — rows whose sample tag starts with this prefix are treated as CRMs.
- `ALLOW_CRM_FLAG_COL` — also accept rows where `CRM_OR_SAMPLE_COL == "crm"`.
- `GROUP_BY` — column name to group corrections by, or `None`.
- `MIN_CRM_N` — minimum number of non-outlier CRMs required.
- `TA_MAD_K` — MAD multiplier for the outlier rule.
- `TA_MAX_ABS_DIFF` — absolute-difference outlier cap; `0` disables this rule.
- `CORRECT_CRM_TOO` — if `False` (default), correction is applied to samples only.
- `TA_NO_ADJUST` — SOP "noise floor": `|corr| <= this` is forced to `0`.
- `TA_REJECT` — SOP reject: `|corr| > this` withholds the row and flags `FAIL`.

**pH standard QC**

- `PH_BUFFER` — `"tris"`, `"amp"`, or `"bis"`.
- `PHSTD_TAG_PREFIX` — rows whose sample tag starts with this prefix are pH standards.
- `PH_MAX_ABS_DIFF` — absolute-difference cap; `0` disables this rule.
- `PH_OK` — status `OK` when `|mean diff| <= this`.
- `PH_WARN` — status `WARN` when `|mean diff| <= this`; `FAIL` otherwise.


## Setup

Dependencies are managed by `pyproject.toml` / `requirements.txt`, not installed
inside the notebook. Install the project environment before running this notebook:

```bash
python -m pip install -e ".[all]"
```

Shared helpers come from `oa_pipeline.common`; the QC functions come from
`oa_pipeline.qc_ta_ph`.


In [ ]:
from __future__ import annotations

import json
import sys
import webbrowser
from pathlib import Path

import pandas as pd

# AUDIT FIX NB02-3: numpy/matplotlib/openpyxl are imported lazily (inside
# the manifest cell and the plotting helpers) so a CSV-only or headless
# run does not hard-crash if matplotlib is absent, matching the README's
# stated 'matplotlib imported lazily' design.

try:
    from IPython.display import display
except Exception:
    display = None

from oa_pipeline.common import (
    die,
    fmt,
    normalize_columns,
    print_quick_summary,
    read_excel_sheets,
    resolve_col,
    safe_sheet_name,
    utc_stamp,
    write_html_table,
    write_manifest,
)

# AUDIT FIX NB02-2: the helper in oa_pipeline.common is named load_config,
# not load_config_file. The previous name silently failed (ImportError
# swallowed) so the shared loader was never used. Import the real name.
try:
    from oa_pipeline.common import load_config as _common_load_config_file
except Exception:
    _common_load_config_file = None

from oa_pipeline.qc_ta_ph import (
    TaSop,
    PhStdStatusThresholds,
    apply_ta_crm_correction,
    load_crm_certified_values,  # AUDIT FIX N-1
    apply_ph_standard_qc_and_correction,
    write_rm_ta_diff_qc_plot,
    write_phstd_qc_plot,
    write_ta_markdown_report,
    write_phstd_markdown_report,
)


def _is_none_like(value) -> bool:
    """Return True for None-like parameter values passed by Papermill or YAML."""
    return value is None or str(value).strip().lower() in {"", "none", "null"}


def _as_bool(value) -> bool:
    """Convert common notebook/config boolean spellings to bool."""
    if isinstance(value, bool):
        return value

    if value is None or pd.isna(value):
        return False

    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1", "on"}:
        return True
    if text in {"false", "f", "no", "n", "0", "off", ""}:
        return False

    die(f"Cannot interpret boolean parameter value: {value!r}")


def _load_notebook_config(config_path: str | Path) -> dict:
    """Load a JSON/YAML config file for this notebook."""
    path = Path(config_path).expanduser().resolve()

    if not path.exists():
        die(f"CONFIG_PATH not found: {path}")

    if _common_load_config_file is not None:
        loaded = _common_load_config_file(path)
        return loaded or {}

    suffix = path.suffix.lower()

    if suffix == ".json":
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle) or {}

    if suffix in {".yaml", ".yml"}:
        try:
            import yaml
        except Exception as exc:
            die(
                "PyYAML is required to read YAML CONFIG_PATH files. "
                "Install with: python -m pip install -e \".[yaml]\". "
                f"Details: {exc}"
            )

        with path.open("r", encoding="utf-8") as handle:
            return yaml.safe_load(handle) or {}

    die(f"Unsupported CONFIG_PATH extension: {path.suffix}. Use .json, .yaml, or .yml.")


## Resolve workbook path and load sheets

In [ ]:
# ---- Resolve input workbook first ------------------------------------
xlsx_path = Path(XLSX_PATH).expanduser().resolve()
if not xlsx_path.exists():
    die(f"File not found: {xlsx_path}")
if xlsx_path.suffix.lower() != ".xlsx":
    die(f"Expected .xlsx, got: {xlsx_path.name}")

# ---- Optional per-stage config ---------------------------------------
config: dict = {}
config_path_resolved = None

if not _is_none_like(CONFIG_PATH):
    config_path_resolved = str(Path(CONFIG_PATH).expanduser().resolve())
    config = _load_notebook_config(CONFIG_PATH)

    parameters = config.get("parameters", {})
    if parameters is None:
        parameters = {}
    if not isinstance(parameters, dict):
        die("CONFIG_PATH must contain a mapping at key 'parameters'.")

    for key, value in parameters.items():
        if key in globals():
            globals()[key] = value
        else:
            print(f"WARNING: unknown config parameter ignored: {key}")

# ---- Normalise parameter types after Papermill/config overrides -------
OPEN_HTML = _as_bool(OPEN_HTML)
WRITE_DERIVED_CSV = _as_bool(WRITE_DERIVED_CSV)
WRITE_HTML_TABLE = _as_bool(WRITE_HTML_TABLE)
WRITE_PLOTS = _as_bool(WRITE_PLOTS)

CRM_CORRECT_TA = _as_bool(CRM_CORRECT_TA)
ALLOW_CRM_FLAG_COL = _as_bool(ALLOW_CRM_FLAG_COL)
NO_REQUIRE_TA_FOR_CRM = _as_bool(NO_REQUIRE_TA_FOR_CRM)
CORRECT_CRM_TOO = _as_bool(CORRECT_CRM_TOO)

PHSTD_QC = _as_bool(PHSTD_QC)
PHSTD_CORRECT_SAMPLES = _as_bool(PHSTD_CORRECT_SAMPLES)
TA_PLOT_NO_LABELS = _as_bool(TA_PLOT_NO_LABELS)
PH_PLOT_NO_LABELS = _as_bool(PH_PLOT_NO_LABELS)

PREVIEW_ROWS = int(PREVIEW_ROWS)
HTML_MAX_ROWS = None if _is_none_like(HTML_MAX_ROWS) else int(HTML_MAX_ROWS)

PH_BUFFER = str(PH_BUFFER).strip().lower()
buffer_slug = safe_sheet_name(PH_BUFFER)

# ---- Validate parameters before any QC work --------------------------
if int(MIN_CRM_N) < 1:
    die(f"MIN_CRM_N must be >= 1, got {MIN_CRM_N}")

if int(MIN_PHSTD_N) < 1:
    die(f"MIN_PHSTD_N must be >= 1, got {MIN_PHSTD_N}")

if float(TA_REJECT) <= float(TA_NO_ADJUST):
    die(
        f"TA_REJECT must be greater than TA_NO_ADJUST. "
        f"Got TA_REJECT={TA_REJECT}, TA_NO_ADJUST={TA_NO_ADJUST}"
    )

if float(PH_WARN) <= float(PH_OK):
    die(
        f"PH_WARN must be greater than PH_OK. "
        f"Got PH_WARN={PH_WARN}, PH_OK={PH_OK}"
    )

if PH_BUFFER not in {"tris", "amp", "bis"}:
    die(f"PH_BUFFER must be one of tris, amp, or bis. Got: {PH_BUFFER}")

if not WRITE_DERIVED_CSV:
    die("WRITE_DERIVED_CSV must be True when running the full pipeline.")

# ---- Sheet selection and output root ---------------------------------
if str(SHEET).strip().lower() == "all":
    sheet_param: "str | int" = "all"
else:
    try:
        sheet_param = int(SHEET)
    except ValueError:
        sheet_param = SHEET

sheets = read_excel_sheets(xlsx_path, sheet_param)

if OUT_DIR:
    out_root = Path(OUT_DIR).expanduser().resolve()
else:
    out_root = xlsx_path.parent / f"{xlsx_path.stem}__qc_outputs"
out_root.mkdir(parents=True, exist_ok=True)

print(f"Workbook   : {xlsx_path}")
print(f"Sheets     : {list(sheets.keys())}")
print(f"Output root: {out_root}")
print(f"Config     : {config_path_resolved or '(built-in defaults)'}")


## Quick preview of each sheet before QC

In [ ]:
for sheet_name, df in sheets.items():
    df = normalize_columns(df)
    print_quick_summary(df, sheet_name, preview_rows=PREVIEW_ROWS)
    print("\n" + "=" * 80 + "\n")


## Run TA CRM QC + pH-standard QC per sheet

For each sheet:

1. Apply TA CRM correction (if `CRM_CORRECT_TA`).
2. Apply pH standard QC (if `PHSTD_QC`).
3. Write `data/derived.csv` (the QC-corrected sheet — this is what Stage 1A reads).
4. Write QC tables, correction tables, JPEG plots, and markdown reports.

Filenames are intentionally short. The sheet identity is in the parent
folder (`sheet_<safe_sheet>/`), so we do not repeat the workbook stem or
"stage 02" tag in every file. The pH-buffer infix (`tris`/`amp`/`bis`) is
kept because it is a meaningful variant — running with two buffers in the
same `out_root` should not overwrite outputs.


In [ ]:
# ---- Translate parameters into dataclass instances ----
ta_max_abs = (
    None
    if _is_none_like(TA_MAX_ABS_DIFF) or float(TA_MAX_ABS_DIFF) == 0
    else float(TA_MAX_ABS_DIFF)
)
ph_max_abs = (
    None
    if _is_none_like(PH_MAX_ABS_DIFF) or float(PH_MAX_ABS_DIFF) == 0
    else float(PH_MAX_ABS_DIFF)
)

sop = TaSop(no_adjust=float(TA_NO_ADJUST), reject=float(TA_REJECT))
ph_thr = PhStdStatusThresholds(ok=float(PH_OK), warn=float(PH_WARN))

# AUDIT FIX N-1: load certified CRM TA values from the authoritative,
# versioned YAML (transcribed from the NOAA OCADS Dickson batch table).
# Passing None falls back to the corrected in-code table. Resolving the
# path relative to CWD or the workbook folder keeps notebook runs working
# whether launched from the repo root or via run_pipeline.sh.
crm_values = None
if CRM_CORRECT_TA and not _is_none_like(CRM_VALUES_CONFIG):
    _crm_cfg = Path(CRM_VALUES_CONFIG).expanduser()
    if not _crm_cfg.is_absolute() and not _crm_cfg.exists():
        # try relative to the workbook's parent as a convenience
        _alt = xlsx_path.parent / CRM_VALUES_CONFIG
        if _alt.exists():
            _crm_cfg = _alt
    crm_values = load_crm_certified_values(_crm_cfg)
    print(f'Loaded {len(crm_values)} certified CRM values from: {_crm_cfg}')
else:
    print('Using corrected in-code CRM fallback table (CRM_VALUES_CONFIG not set).')

# Snapshot of params for the markdown reports
report_params = {
    "CRM_OR_SAMPLE_COL": CRM_OR_SAMPLE_COL,
    "CRM_TAG_PREFIX": CRM_TAG_PREFIX,
    "CRM_BATCH": CRM_BATCH,
    "PH_BUFFER": PH_BUFFER,
    "PHSTD_TAG_PREFIX": PHSTD_TAG_PREFIX,
}

written_files: list[Path] = []
sheet_summaries: dict[str, dict] = {}

for sheet_name, df in sheets.items():
    df = normalize_columns(df)
    safe_sheet = safe_sheet_name(str(sheet_name))

    sheet_root = out_root / f"sheet_{safe_sheet}"
    data_dir = sheet_root / "data"
    qc_dir = sheet_root / "qc"
    fig_dir = sheet_root / "figures"
    rep_dir = sheet_root / "reports"
    tab_dir = sheet_root / "tables"
    for d in (data_dir, qc_dir, fig_dir, rep_dir, tab_dir):
        d.mkdir(parents=True, exist_ok=True)

    derived_csv = data_dir / "derived.csv"
    html_path = tab_dir / "table.html"

    df_out = df.copy()
    ta_summary: dict = {}
    ph_summary: dict = {}

    # ---- TA CRM QC --------------------------------------------------
    if CRM_CORRECT_TA:
        df_out, crm_qc, ta_corr_table, ta_summary = apply_ta_crm_correction(
            df=df_out,
            ta_col=TA_COL,
            sample_tag_col=SAMPLE_TAG_COL,
            crm_or_sample_col=CRM_OR_SAMPLE_COL,
            crm_batch=CRM_BATCH,
            crm_ta_override=CRM_TA_OVERRIDE,
            group_by=GROUP_BY,
            crm_tag_prefix=CRM_TAG_PREFIX,
            allow_crm_flag_col=ALLOW_CRM_FLAG_COL,
            require_ta_value_for_crm=not NO_REQUIRE_TA_FOR_CRM,
            min_crm_n=int(MIN_CRM_N),
            mad_k=float(TA_MAD_K),
            max_abs_diff=ta_max_abs,
            correct_only_samples=not CORRECT_CRM_TOO,
            sop=sop,
            crm_values=crm_values,  # AUDIT FIX N-1
        )

        ta_qc_csv = qc_dir / "ta_crm_qc.csv"
        ta_corr_csv = qc_dir / "ta_corrections.csv"
        ta_md = rep_dir / "ta_crm_report.md"

        crm_qc.to_csv(ta_qc_csv, index=False)
        ta_corr_table.to_csv(ta_corr_csv, index=False)
        write_ta_markdown_report(
            out_md=ta_md,
            xlsx_path=xlsx_path,
            sheet_name=str(sheet_name),
            params=report_params,
            ta_summary=ta_summary,
            ta_col_used=resolve_col(df, TA_COL),
            sample_tag_col_used=resolve_col(df, SAMPLE_TAG_COL),
            group_by=GROUP_BY,
        )
        written_files.extend([ta_qc_csv, ta_corr_csv, ta_md])

        print(
            f"[TA CRM] sheet={sheet_name} "
            f"status={ta_summary.get('overall_status')} "
            f"detected={ta_summary.get('crm_n_detected')} "
            f"kept={ta_summary.get('crm_n_kept')} "
            f"mean_diff={fmt(ta_summary.get('overall_corr'), nd=3)}"
        )

        if WRITE_PLOTS:
            ta_jpeg = fig_dir / "rm_ta_diff_qc.jpeg"
            tag_col_in_crm = SAMPLE_TAG_COL if SAMPLE_TAG_COL in crm_qc.columns else None
            if tag_col_in_crm is not None:
                write_rm_ta_diff_qc_plot(
                    crm_qc=crm_qc,
                    out_jpeg=ta_jpeg,
                    sample_tag_col=resolve_col(crm_qc, tag_col_in_crm),
                    sop=sop,
                    annotate_points=not TA_PLOT_NO_LABELS,
                    title=str(TA_PLOT_TITLE),
                )
                written_files.append(ta_jpeg)

    # ---- pH standard QC --------------------------------------------
    if PHSTD_QC:
        df_out, phstd_qc, ph_corr_table, ph_summary = apply_ph_standard_qc_and_correction(
            df=df_out,
            buffer=PH_BUFFER,
            tag_prefix=PHSTD_TAG_PREFIX,
            ph_col=PH_COL,
            temp_col=PH_TEMP_COL,
            sample_tag_col=SAMPLE_TAG_COL,
            crm_or_sample_col=CRM_OR_SAMPLE_COL,
            group_by=GROUP_BY,
            mad_k=float(PH_MAD_K),
            max_abs_diff=ph_max_abs,
            min_std_n=int(MIN_PHSTD_N),
            correct_samples=PHSTD_CORRECT_SAMPLES,
            status_thr=ph_thr,
        )

        ph_qc_csv = qc_dir / f"phstd_qc_{buffer_slug}.csv"
        ph_corr_csv = qc_dir / f"phstd_corrections_{buffer_slug}.csv"
        ph_md = rep_dir / f"phstd_report_{buffer_slug}.md"

        phstd_qc.to_csv(ph_qc_csv, index=False)
        ph_corr_table.to_csv(ph_corr_csv, index=False)
        write_phstd_markdown_report(
            out_md=ph_md,
            xlsx_path=xlsx_path,
            sheet_name=str(sheet_name),
            params=report_params,
            ph_summary=ph_summary,
        )
        written_files.extend([ph_qc_csv, ph_corr_csv, ph_md])

        print(
            f"[pH STD] sheet={sheet_name} "
            f"status={ph_summary.get('overall_status')} "
            f"detected={ph_summary.get('n_detected')} "
            f"kept={ph_summary.get('n_kept')} "
            f"mean_diff={fmt(ph_summary.get('mean_diff_kept'), nd=4)}"
        )

        # Only attempt the pH-standard plot when there are usable standards
        # (numeric expected-minus-measured diffs). With kept=0 / all-NaN
        # diffs the plot has nothing to draw; skip with a warning rather than
        # letting the plot helper hard-exit the whole pipeline.
        _ph_has_plottable = (
            "phstd_diff" in phstd_qc.columns
            and pd.to_numeric(phstd_qc["phstd_diff"], errors="coerce").notna().any()
        )
        if WRITE_PLOTS and not _ph_has_plottable:
            print(
                f"[pH STD] sheet={sheet_name} no usable standard diffs "
                f"(kept={ph_summary.get('n_kept')}); skipping QC plot."
            )
        if WRITE_PLOTS and _ph_has_plottable:
            ph_jpeg = fig_dir / f"phstd_diff_qc_{buffer_slug}.jpeg"
            tag_col_in_std = SAMPLE_TAG_COL if SAMPLE_TAG_COL in phstd_qc.columns else None
            if tag_col_in_std is not None:
                write_phstd_qc_plot(
                    phstd_qc=phstd_qc,
                    out_jpeg=ph_jpeg,
                    sample_tag_col=resolve_col(phstd_qc, tag_col_in_std),
                    diff_col="phstd_diff",
                    outlier_col="phstd_diff_is_outlier",
                    thr_ok=float(ph_thr.ok),
                    thr_warn=float(ph_thr.warn),
                    annotate_points=not PH_PLOT_NO_LABELS,
                    title=str(PH_PLOT_TITLE),
                    summary=ph_summary,
                )
                written_files.append(ph_jpeg)

    # ---- Derived CSV (the file Stage 1A reads) + HTML table --------
    if WRITE_DERIVED_CSV:
        df_out.to_csv(derived_csv, index=False)
        written_files.append(derived_csv)
        print(f"Wrote derived CSV: {derived_csv}")

    if WRITE_HTML_TABLE:
        write_html_table(
            df_out,
            html_path,
            max_rows=HTML_MAX_ROWS,
            title=f"{xlsx_path.stem} - sheet {sheet_name} (derived)",
        )
        written_files.append(html_path)
        if OPEN_HTML:
            webbrowser.open(html_path.as_uri())

    sheet_summaries[str(sheet_name)] = {
        "sheet_name": str(sheet_name),
        "sheet_safe_name": safe_sheet,
        "ta_summary": ta_summary,
        "ph_summary": ph_summary,
        "root": str(sheet_root),
        "derived_csv": str(derived_csv) if WRITE_DERIVED_CSV else None,
    }


## Summary of written files and per-sheet QC status

In [ ]:
outputs_df = pd.DataFrame({"output_file": [str(p) for p in written_files]})
print(f"Total files written: {len(outputs_df)}")
if display is not None:
    display(outputs_df)
else:
    print(outputs_df.to_string(index=False))

print("\nPer-sheet summary:\n")
for sheet_name, info in sheet_summaries.items():
    print(f"Sheet: {sheet_name}")
    ta = info.get("ta_summary") or {}
    if ta:
        print(
            "  TA CRM:",
            f"status={ta.get('overall_status')},",
            f"detected={ta.get('crm_n_detected')},",
            f"kept={ta.get('crm_n_kept')},",
            f"mean_diff={fmt(ta.get('overall_corr'), nd=3)}",
        )
    ph = info.get("ph_summary") or {}
    if ph:
        print(
            "  pH STD:",
            f"status={ph.get('overall_status')},",
            f"detected={ph.get('n_detected')},",
            f"kept={ph.get('n_kept')},",
            f"mean_diff={fmt(ph.get('mean_diff_kept'), nd=4)}",
        )
    print(f"  Safe sheet name: {info['sheet_safe_name']}")
    print(f"  Derived CSV    : {info.get('derived_csv')}")
    print(f"  Output root    : {info['root']}")


## Manifest (provenance log)

Same pattern as Notebook 01: a single `manifest.json` under `logs/` that
records the inputs, parameters, outputs and package versions. Provenance
lives here, not in filenames.


In [ ]:
logs_dir = out_root / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)
manifest_path = logs_dir / "manifest.json"

derived_csv_by_sheet = {
    sheet: info.get("derived_csv")
    for sheet, info in sheet_summaries.items()
}

def _collect_package_versions() -> dict:
    """Best-effort capture of package versions without hard import deps."""
    versions = {"python": sys.version.split()[0], "pandas": pd.__version__}
    for name, mod in (("numpy", "numpy"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")):
        try:
            import importlib
            versions[name] = importlib.import_module(mod).__version__
        except Exception:
            versions[name] = None
    return versions


manifest = {
    "notebook": "02_ta_ph_qc",
    "generated_utc": utc_stamp(),
    "input_xlsx": str(xlsx_path),
    "output_root": str(out_root),
    "config": {
        "config_path": config_path_resolved,
        "parameter_keys": sorted(list((config.get("parameters") or {}).keys())),
    },
    "parameters": {
        # I/O
        "XLSX_PATH": str(xlsx_path),
        "OUT_DIR": str(out_root),
        "SHEET": SHEET,
        "CONFIG_PATH": config_path_resolved,
        # display
        "PREVIEW_ROWS": PREVIEW_ROWS,
        "HTML_MAX_ROWS": HTML_MAX_ROWS,
        "OPEN_HTML": OPEN_HTML,
        "WRITE_DERIVED_CSV": WRITE_DERIVED_CSV,
        "WRITE_HTML_TABLE": WRITE_HTML_TABLE,
        "WRITE_PLOTS": WRITE_PLOTS,
        # columns
        "TA_COL": TA_COL,
        "PH_COL": PH_COL,
        "PH_TEMP_COL": PH_TEMP_COL,
        "SAMPLE_TAG_COL": SAMPLE_TAG_COL,
        "CRM_OR_SAMPLE_COL": CRM_OR_SAMPLE_COL,
        # TA
        "CRM_CORRECT_TA": CRM_CORRECT_TA,
        "CRM_BATCH": CRM_BATCH,
        "CRM_VALUES_CONFIG": str(CRM_VALUES_CONFIG),
        "CRM_VALUES_SOURCE": ("yaml_file" if crm_values is not None else "in_code_fallback"),
        "CRM_TA_OVERRIDE": CRM_TA_OVERRIDE,
        "CRM_TAG_PREFIX": CRM_TAG_PREFIX,
        "ALLOW_CRM_FLAG_COL": ALLOW_CRM_FLAG_COL,
        "NO_REQUIRE_TA_FOR_CRM": NO_REQUIRE_TA_FOR_CRM,
        "GROUP_BY": GROUP_BY,
        "MIN_CRM_N": MIN_CRM_N,
        "TA_MAD_K": TA_MAD_K,
        "TA_MAX_ABS_DIFF": TA_MAX_ABS_DIFF,
        "CORRECT_CRM_TOO": CORRECT_CRM_TOO,
        "TA_NO_ADJUST": TA_NO_ADJUST,
        "TA_REJECT": TA_REJECT,
        "TA_PLOT_TITLE": TA_PLOT_TITLE,
        "TA_PLOT_NO_LABELS": TA_PLOT_NO_LABELS,
        # pH
        "PHSTD_QC": PHSTD_QC,
        "PH_BUFFER": PH_BUFFER,
        "PH_BUFFER_FILE_SLUG": buffer_slug,
        "PHSTD_TAG_PREFIX": PHSTD_TAG_PREFIX,
        "PHSTD_CORRECT_SAMPLES": PHSTD_CORRECT_SAMPLES,
        "MIN_PHSTD_N": MIN_PHSTD_N,
        "PH_MAD_K": PH_MAD_K,
        "PH_MAX_ABS_DIFF": PH_MAX_ABS_DIFF,
        "PH_OK": PH_OK,
        "PH_WARN": PH_WARN,
        "PH_PLOT_TITLE": PH_PLOT_TITLE,
        "PH_PLOT_NO_LABELS": PH_PLOT_NO_LABELS,
    },
    "sheets_processed": list(sheets.keys()),
    "derived_csv_by_sheet": derived_csv_by_sheet,
    "outputs": [str(p) for p in written_files],
    "sheet_summaries": sheet_summaries,
    # AUDIT FIX NB02-3: import version-only modules lazily here so the
    # main run does not require them at import time.
    "package_versions": _collect_package_versions(),
}

write_manifest(manifest_path, manifest)
print(f"Wrote manifest: {manifest_path}")
